# 01 · Welcome and a tour of the stack

**Time:** ~15 min · **You will learn:** what each container does, how to talk to Iceberg from PySpark and from PyIceberg.

## What is Apache Iceberg?
Iceberg is an **open table format**. It is not a database or a query engine: it is a specification for how to lay out
data files (Parquet) and metadata files in object storage so that many engines can treat that pile of files as a
reliable **table** with transactions, schema changes and time travel.

## The stack you are running
```
+---------------------+      +---------------------+      +--------------------+
|  Jupyter + PySpark  | ---> |  Iceberg REST       | ---> |  RustFS (S3 API)   |
|  PyIceberg          |      |  catalog  :8181     |      |  bucket: warehouse |
|  (spark-iceberg)    |      |  "which tables      |      |  Parquet + JSON    |
+---------------------+      |   exist, where is   |      |  metadata files    |
          |                  |   their metadata?"  |      +--------------------+
          +------------------+---------------------+              ^
                     data files are read and written directly -----+
```

* **Catalog** (REST service): the phone book. It maps `course.orders` to the location of the table's latest metadata file.
* **Object storage** (RustFS): holds the data files and metadata files. Engines read and write it directly.
* **Engines**: PySpark (big, distributed work) and PyIceberg (a lightweight Python library, no cluster needed).

Useful URLs: Jupyter http://localhost:8888, Spark UI http://localhost:8080, REST catalog http://localhost:8181/v1/config,
RustFS console http://localhost:9001 (login `admin` / `password`).

## 1. Start Spark
The helper below creates a Spark session and makes sure our `course` namespace exists.

In [1]:
from helpers import get_spark

spark = get_spark("01-tour")
print("Spark version:", spark.version)
print("Default catalog:", spark.conf.get("spark.sql.defaultCatalog"))

26/10/17 01:30:57 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Spark version: 3.5.6
Default catalog: demo


The catalog is called **`demo`** (see `spark/spark-defaults.conf`). A table is addressed as
`catalog.namespace.table`, for example `demo.course.orders`. Because `demo` is the default catalog,
`course.orders` also works.

In [2]:
spark.sql("SHOW NAMESPACES IN demo").show()

+---------+
|namespace|
+---------+
|   course|
+---------+



## 2. Talk to the same catalog from PyIceberg
PyIceberg reads its settings from `~/.pyiceberg.yaml` (a copy of `spark/.pyiceberg.yaml`).

In [3]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog("default")
print("Namespaces seen by PyIceberg:", catalog.list_namespaces())

Namespaces seen by PyIceberg: [('course',)]


Spark and PyIceberg are two different programs, but they see the **same** namespaces because they both ask the same REST catalog.
That is the whole point of an open table format.

### Your turn

Create a namespace called `scratch` from **Spark**, then confirm that **PyIceberg** can see it.
Finally drop it again (`DROP NAMESPACE`).

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.scratch")
assert ("scratch",) in catalog.list_namespaces()
print(catalog.list_namespaces())
spark.sql("DROP NAMESPACE demo.scratch")
assert ("scratch",) not in catalog.list_namespaces()
```

</details>

### Recap
* Iceberg = table format on top of files in object storage; a **catalog** tracks the current metadata of each table.
* Tables are addressed as `catalog.namespace.table`.
* PySpark and PyIceberg share tables through the catalog.

Next: **02 · Your first Iceberg table**.